# 09 - Socioeconomic equity and Simpson's paradox

This notebook asks whether Israel's bus and rail network gives a higher level of service to wealthier neighborhoods. Every stop is matched to a Central Bureau of Statistics (CBS) socioeconomic statistical area for 2021, service is aggregated up to the neighborhood level (the statistical area), and the socioeconomic cluster (1 = weakest, 10 = strongest) is correlated with service per capita.

The headline result is Simpson's paradox: nationally the relationship is weak and slightly negative, but within a single metropolitan area (represented here by a Louvain community of the trip graph) the relationship is usually strong - and it points in opposite directions in different metros, so the local effects cancel out when you aggregate.

**Research question.** Does socioeconomic status predict transit service per capita in Israel, and is the national answer the same as the answer inside a single city or metro?

**Input**
- `outputs/nb/<earlier stage>/tables/stop_metrics.csv` - stop-level metrics from the graph-building / centrality stage (stop coordinates, `stop_use_count`, `degree` of trip adjacency, Louvain community, betweenness, articulation-point flag). There is a fallback to the canonical repository file `outputs/tables/stop_metrics.csv`.
- CBS 2021 socioeconomic statistical areas, downloaded at run time from an ArcGIS REST service (external dependency - see the warning below).

**Output** (all under `outputs/nb/09_socioeconomic_equity/`)
- `tables/stops_with_socioeconomic.csv`, `tables/socioeconomic_neighborhood_access.csv`
- `tables/socioeconomic_cluster_summary.csv`, `tables/community_socioeconomic_summary.csv`
- `tables/socioeconomic_national_correlations.csv`, `tables/socioeconomic_within_cluster_correlation.csv`
- `tables/trend_fit_sensitivity.csv`, `tables/socioeconomic_join_quality.json`, `tables/socioeconomic_summary.json`
- `data/cbs_socioeconomic_areas_2021.geojson` (a cached download, so repeat runs don't need the network)
- `figures/socioeconomic_access_by_cluster.png`, `figures/socioeconomic_cluster_average_flat.png`, `figures/socioeconomic_cities_no_rule.png`, `figures/socioeconomic_within_cluster_correlation.png`, `figures/socioeconomic_within_cluster_examples.png`, `figures/socioeconomic_simpson_paradox.png`

**Run time.** A few minutes end to end. The two expensive steps are downloading the CBS polygons (about 2,900 polygons, cached after the first run) and the point-in-polygon join of about 30,000 stops.

**A note on terminology.** The word cluster is used for two different things in the sources, so this notebook keeps the distinction sharp:
- socioeconomic cluster = the CBS 1-10 index of a statistical area;
- community = a Louvain community of the public-transport graph, used here as a data-driven proxy for a city / metro.

The published figure filenames keep the original wording (`within_cluster`) so they stay comparable, but the axis labels say community.

In [1]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

Repo root: C:\Users\Sean-PC\Desktop\LAST SEMESTER\אלגוריתמים ברשתות\פרויקט גמר


## Stage folders and tunable constants

Everything this notebook writes lives in its own dedicated stage folder, so it can't overwrite results in `outputs/tables`, `outputs/figures`, or `outputs/rail`.

The constants below are gathered here because they're the knobs that change the result. These are the ones worth knowing:

- `NEAREST_JOIN_MAX_DISTANCE_M` - a stop that doesn't fall inside any CBS polygon (rural roads, interchanges, industrial zones) is matched to the nearest polygon up to this distance. Beyond it the stop stays unmatched, rather than being forced into a neighborhood it doesn't belong to.
- `MIN_NEIGHBORHOODS` / `MIN_DISTINCT_SOCIO_LEVELS` - a within-community correlation is computed only when the community has enough neighborhoods, spread across enough distinct socioeconomic levels, for the rank correlation to carry information.
- `DISPLAY_TRIM_PCT` - used only to trim the y-axis display in the scatter plots. It's never used to fit a line or to compute a statistic (see the note on the trimming bug below).
- `CITIES_IN_NO_RULE_FIGURE` - left as `None` so the illustration cities are chosen from the data rather than hard-coded.

In [2]:
STAGE = OUT / '09_socioeconomic_equity'
(STAGE / 'tables').mkdir(parents=True, exist_ok=True)
(STAGE / 'figures').mkdir(parents=True, exist_ok=True)
(STAGE / 'data').mkdir(parents=True, exist_ok=True)

# --- External data source: CBS 2021 socioeconomic statistical areas (ArcGIS REST) ---
CBS_LAYER_URL = ('https://services2.arcgis.com/xMRYm7cNgdR5RN6F/arcgis/rest/services/'
                 'SOEC_Stat11_2021/FeatureServer/27')
CBS_PAGE_SIZE = 2000                      # features per request (the service caps a page at 2000)
CBS_CACHE = STAGE / 'data' / 'cbs_socioeconomic_areas_2021.geojson'

# Spatial join fallback distance, in metres (measured in EPSG:3857).
NEAREST_JOIN_MAX_DISTANCE_M = 3000

# A stop counts as critical if it is an articulation point or sits in the top
# decile of betweenness - the same rule the rest of the project uses.
CRITICAL_BETWEENNESS_QUANTILE = 0.90

# Within-community test thresholds.
MIN_NEIGHBORHOODS = 12
MIN_DISTINCT_SOCIO_LEVELS = 3
ALPHA = 0.05

# Figure-only thresholds. The saved tables keep every community that passes the
# thresholds above; the figures show a readable subset of them.
FIG_MIN_NEIGHBORHOODS = 20
TOP_COMMUNITIES_FOR_FIGURE = 15
EXAMPLE_MIN_NEIGHBORHOODS = 40
DISPLAY_TRIM_PCT = 90        # y-axis VIEW clipping only - never used for fitting or statistics

# Illustrative cities for the bar chart. None -> derive from the data.
CITIES_IN_NO_RULE_FIGURE = None
N_CITIES_NO_RULE = 6

print('Stage folder :', STAGE)
print('CBS cache    :', CBS_CACHE)

Stage folder : C:\Users\Sean-PC\Desktop\LAST SEMESTER\אלגוריתמים ברשתות\פרויקט גמר\outputs\nb\09_socioeconomic_equity
CBS cache    : C:\Users\Sean-PC\Desktop\LAST SEMESTER\אלגוריתמים ברשתות\פרויקט גמר\outputs\nb\09_socioeconomic_equity\data\cbs_socioeconomic_areas_2021.geojson


## Hebrew labels in matplotlib

The town names come from the CBS layer in Hebrew, and they show up in axis labels and figure legends. matplotlib doesn't run the Unicode bidirectional algorithm, so Hebrew strings display reversed. The fix below rewrites the text into display order once, before any figure is drawn. Latin text passes through unchanged, so English axis titles aren't affected.

In [3]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

# Fail loudly if python-bidi is not actually reordering Hebrew. A broken or no-op
# build makes get_display a pass-through, silently rendering every Hebrew label
# reversed. Assert the reordering works so the notebook errors here instead of
# producing backwards figures.
if get_display("אבג") != "גבא":
    raise RuntimeError(
        "python-bidi is not reordering Hebrew (get_display is a no-op); Hebrew "
        "figure labels would render reversed. Reinstall a working build:\n"
        "    pip install -U --force-reinstall 'python-bidi>=0.6.7'"
    )

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## Libraries

`geopandas` (together with `shapely` and `pyproj`) does the point-in-polygon and nearest-polygon joins; `requests` pages through the ArcGIS service; `scipy` provides the Spearman rank correlation. `seaborn` is used only for the chart theme, and `install_hebrew()` is called again afterward because `sns.set_theme` resets the font family.

In [4]:
_ensure('geopandas', 'requests', 'scipy', 'seaborn')

import json
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', font_scale=1.0)
install_hebrew()   # sns.set_theme resets the font family; re-apply the Hebrew-capable fonts

print('pandas', pd.__version__, '| geopandas', gpd.__version__)

pandas 2.3.3 | geopandas 1.1.1


## Sign convention (read this before looking at any number)

A review of the previous version found that the presentation text quoted the raw Spearman rho, while the chart plotted `equity_bias = -rho`, so the same city was reported as `+0.48` in one place and `-0.48` in another.

This notebook uses the raw Spearman rho everywhere - in the tables, the axis labels, and the text. No sign is flipped. Since the socioeconomic cluster runs from 1 (weakest) to 10 (strongest):

| rho | meaning |
| --- | --- |
| **rho < 0** | service per capita is higher in weaker neighborhoods |
| **rho > 0** | service per capita is higher in stronger neighborhoods |

Color is only a redundant cue for that same direction (green = negative rho, red = positive rho); the number printed on each bar is the raw rho.

### Second convention: fits and statistics have to agree with each other

The previous version drew trend lines fitted on percentile-trimmed data, while it quoted the untrimmed Spearman as the headline statistic, so the line and the number described different datasets. Here every trend line is an ordinary least squares (OLS) fit on exactly the same untrimmed rows that go into the reported Spearman. Percentile trimming survives only as a display definition of the y-axis, and `tables/trend_fit_sensitivity.csv` records how much the slope would have changed if the old trimming had been applied, so the choice is auditable and not hidden.

In [5]:
RHO_AXIS_LABEL = 'Spearman rho  (socioeconomic cluster  vs  service per 1,000 residents)'
RHO_NEG_MEANING = 'rho < 0  ->  more service per capita in the WEAKER neighbourhoods'
RHO_POS_MEANING = 'rho > 0  ->  more service per capita in the STRONGER neighbourhoods'

NEG_COLOR = '#1e8449'   # green: negative rho
POS_COLOR = '#c0392b'   # red:   positive rho

def rho_color(rho):
    """Colour encodes the SIGN OF THE RAW RHO - no sign flipping anywhere."""
    return NEG_COLOR if rho < 0 else POS_COLOR

def describe_rho(rho):
    if rho is None or (isinstance(rho, float) and np.isnan(rho)):
        return 'undefined'
    return 'weaker neighbourhoods favoured' if rho < 0 else 'stronger neighbourhoods favoured'

def spearman(x, y, min_n=8):
    """Spearman rank correlation on the raw (untrimmed) overlapping rows.

    Returns (rho, p_value, n). Returns NaNs when the sample is too small or one
    of the variables is constant, so the caller never sees a meaningless 1.0.
    """
    sample = pd.DataFrame({'x': pd.to_numeric(pd.Series(x).reset_index(drop=True), errors='coerce'),
                           'y': pd.to_numeric(pd.Series(y).reset_index(drop=True), errors='coerce')}).dropna()
    if len(sample) < min_n or sample['x'].nunique() < 2 or sample['y'].nunique() < 2:
        return np.nan, np.nan, len(sample)
    rho, p_value = spearmanr(sample['x'], sample['y'])
    return float(rho), float(p_value), int(len(sample))

def ols_slope(x, y):
    """Least-squares slope/intercept on the untrimmed overlapping rows."""
    sample = pd.DataFrame({'x': pd.to_numeric(pd.Series(x).reset_index(drop=True), errors='coerce'),
                           'y': pd.to_numeric(pd.Series(y).reset_index(drop=True), errors='coerce')}).dropna()
    if len(sample) < 2 or sample['x'].nunique() < 2:
        return None
    return np.polyfit(sample['x'], sample['y'], 1)

def per_1000(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors='coerce')
    denominator = pd.to_numeric(denominator, errors='coerce')
    return np.where(denominator > 0, numerator / denominator * 1000, np.nan)

def mode_or_na(series):
    values = series.dropna()
    return values.mode().iloc[0] if len(values) else np.nan

def save_fig(fig, name):
    path = STAGE / 'figures' / name
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    print('saved', path.name)
    return path

print(RHO_NEG_MEANING)
print(RHO_POS_MEANING)

rho < 0  ->  more service per capita in the WEAKER neighbourhoods
rho > 0  ->  more service per capita in the STRONGER neighbourhoods


## Loading the stop-level graph metrics

This stage doesn't rebuild the graph; it consumes the stop table produced in the graph-building / centrality stage. The loader first looks for `outputs/nb/*/tables/stop_metrics.csv` (that is, an earlier notebook in this series) and falls back to the canonical repository file `outputs/tables/stop_metrics.csv` that ships with the source version. If neither exists, a clear and actionable error is raised instead of quietly analyzing nothing.

**A bug fixed here.** The published `stops_with_socioeconomic.csv` carried a `degree` column inherited from a retired 500 m spatial-proximity graph, and the neighborhood table aggregated it as `avg_proximity_degree`. Proximity degree measures how dense a stop's 500 m surroundings are - it's really a density metric, not a service metric - so it has no place in a network-structure analysis. This notebook takes `degree` from the trip-adjacency graph (two stops are adjacent when a scheduled trip serves them consecutively) and calls it `trip_graph_degree` everywhere, so the obsolete column can't sneak back in.

In [6]:
def find_stop_metrics():
    """Prefer an earlier notebook stage; fall back to the committed pipeline table."""
    candidates = sorted(OUT.glob('*/tables/stop_metrics.csv'))
    canonical = REPO / 'outputs' / 'tables' / 'stop_metrics.csv'
    if canonical.exists():
        candidates.append(canonical)
    if not candidates:
        raise FileNotFoundError(
            'No stop_metrics.csv found under ' + str(OUT) + '/*/tables/ and no '
            + str(canonical) + '. Run the graph-construction / centrality notebook '
            '(e.g. 02_graph_construction) first.')
    return candidates[0]

STOP_METRICS_PATH = find_stop_metrics()
print('Reading per-stop metrics from:', STOP_METRICS_PATH)

metrics = pd.read_csv(STOP_METRICS_PATH, encoding='utf-8-sig', low_memory=False)
metrics.columns = [str(c).strip().lstrip('\ufeff') for c in metrics.columns]

# The Louvain community label is produced by notebook 08, not by the centrality
# stage, so stop_metrics.csv does not carry it. Merge it in when it is absent.
_COMM_NAMES = ['community_id', 'community_louvain', 'community']
if not any(c in metrics.columns for c in _COMM_NAMES):
    _comm_files = sorted(OUT.glob('08*/tables/community_assignments.csv'))
    if not _comm_files:
        raise FileNotFoundError(
            'No community_assignments.csv under ' + str(OUT) + '/08*/tables/. '
            'Run 08_community_detection.ipynb first - this notebook needs Louvain '
            'communities to compute the within-cluster correlations.')
    _comm = pd.read_csv(_comm_files[0], encoding='utf-8-sig', low_memory=False)
    _comm.columns = [str(c).strip() for c in _comm.columns]
    _cc = next((c for c in _COMM_NAMES if c in _comm.columns), None)
    if _cc is None:
        raise KeyError(str(_comm_files[0]) + ' has no community column; found: '
                       + str(list(_comm.columns)))
    metrics['stop_id'] = metrics['stop_id'].astype(str)
    _comm['stop_id'] = _comm['stop_id'].astype(str)
    metrics = metrics.merge(_comm[['stop_id', _cc]], on='stop_id', how='left')
    print('Merged Louvain communities from:', _comm_files[0])
    print('  stops with a community:', int(metrics[_cc].notna().sum()))

def pick_column(frame, names, what, required=True):
    for name in names:
        if name in frame.columns:
            return name
    if required:
        raise KeyError(str(STOP_METRICS_PATH) + ' has none of ' + str(names)
                       + ' (needed for ' + what + ').')
    return None

lat_col = pick_column(metrics, ['stop_lat', 'lat'], 'stop latitude')
lon_col = pick_column(metrics, ['stop_lon', 'lon'], 'stop longitude')
use_col = pick_column(metrics, ['stop_use_count'], 'scheduled stop calls')
deg_col = pick_column(metrics, ['degree'], 'trip-adjacency degree')
com_col = pick_column(metrics, ['community_id', 'community_louvain', 'community'], 'Louvain community')
btw_col = pick_column(metrics, ['approx_betweenness', 'betweenness'], 'betweenness', required=False)

stops = pd.DataFrame({
    'stop_id': metrics['stop_id'].astype(str),
    'stop_name': metrics['stop_name'].astype(str) if 'stop_name' in metrics.columns else '',
    'lat': pd.to_numeric(metrics[lat_col], errors='coerce'),
    'lon': pd.to_numeric(metrics[lon_col], errors='coerce'),
    'stop_use_count': pd.to_numeric(metrics[use_col], errors='coerce').fillna(0.0),
    # NOTE: trip-adjacency degree, NOT the retired 500 m proximity degree.
    'trip_graph_degree': pd.to_numeric(metrics[deg_col], errors='coerce').fillna(0.0),
    'community': pd.to_numeric(metrics[com_col], errors='coerce'),
})
if 'weighted_degree' in metrics.columns:
    stops['trip_graph_weighted_degree'] = pd.to_numeric(metrics['weighted_degree'], errors='coerce').fillna(0.0)
if btw_col is not None:
    stops['betweenness'] = pd.to_numeric(metrics[btw_col], errors='coerce').fillna(0.0)
if 'is_articulation_point' in metrics.columns:
    stops['is_articulation_point'] = (metrics['is_articulation_point'].astype(str)
                                      .str.strip().str.lower().isin(['true', '1', 'yes']))

# Louvain marks unassigned nodes with -1 in some exports; that is missing, not a community.
stops.loc[stops['community'] < 0, 'community'] = np.nan
stops = stops.dropna(subset=['lat', 'lon']).reset_index(drop=True)

print('stops with coordinates      :', len(stops))
print('community coverage          : %.1f%% of stops, %d communities'
      % (100 * stops['community'].notna().mean(), stops['community'].dropna().nunique()))
print('columns kept                :', list(stops.columns))
assert 'proximity' not in ' '.join(stops.columns), 'A proximity-graph column leaked in'

Reading per-stop metrics from: C:\Users\Sean-PC\Desktop\LAST SEMESTER\אלגוריתמים ברשתות\פרויקט גמר\outputs\nb\04_centrality_analysis\tables\stop_metrics.csv
Merged Louvain communities from: C:\Users\Sean-PC\Desktop\LAST SEMESTER\אלגוריתמים ברשתות\פרויקט גמר\outputs\nb\08_community_detection\tables\community_assignments.csv
  stops with a community: 30463
stops with coordinates      : 30463
community coverage          : 99.3% of stops, 73 communities
columns kept                : ['stop_id', 'stop_name', 'lat', 'lon', 'stop_use_count', 'trip_graph_degree', 'community', 'trip_graph_weighted_degree', 'betweenness']


## Region and metro labels

The neighborhood and community tables carry a dominant region and dominant metro, which let you read the results clearly ("this community is the Haifa area"). These are coarse geographic labels derived from stop coordinates - the same rule the project's data-preparation stage used: first a Jerusalem bounding box, then latitude bands for north / center / south, and a radius around each of the four metro centers. They're descriptive only and are never used as a variable in the analysis.

In [7]:
METRO_CENTERS = {
    'תל אביב': (32.0853, 34.7818, 30),
    'חיפה': (32.7940, 34.9896, 25),
    'ירושלים': (31.7683, 35.2137, 20),
    'באר שבע': (31.2518, 34.7913, 25),
}

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
    return 2 * radius * np.arcsin(np.sqrt(a))

def assign_region(lat, lon):
    if 31.70 <= lat <= 31.90 and 34.95 <= lon <= 35.30:
        return 'ירושלים'
    if lat > 32.50:
        return 'צפון'
    if lat >= 31.55:
        return 'מרכז'
    return 'דרום'

def assign_metro(lat, lon):
    for city, (clat, clon, radius) in METRO_CENTERS.items():
        if haversine_km(lat, lon, clat, clon) <= radius:
            return city
    return 'פריפריה'

stops['region'] = [assign_region(a, b) for a, b in zip(stops['lat'], stops['lon'])]
stops['metro'] = [assign_metro(a, b) for a, b in zip(stops['lat'], stops['lon'])]

print(stops['region'].value_counts().to_string())
print()
print(stops['metro'].value_counts().to_string())

region
מרכז       12970
צפון       10025
ירושלים     3743
דרום        3725

metro
פריפריה    12627
תל אביב     8427
חיפה        4023
ירושלים     3474
באר שבע     1912


## Criticality flag

The equity tables report what share of a neighborhood's stops are critical, by the project's definition: a stop is critical if it's an articulation point - removing it disconnects part of the network - or if its betweenness is in the top decile. If those columns are missing from the metrics table, the flag is set to `False` and the share simply becomes uninformative rather than wrong.

In [8]:
if 'betweenness' in stops.columns and 'is_articulation_point' in stops.columns:
    threshold = float(stops['betweenness'].quantile(CRITICAL_BETWEENNESS_QUANTILE))
    stops['is_critical'] = stops['is_articulation_point'] | (stops['betweenness'] >= threshold)
    print('betweenness p%d threshold : %.6g' % (CRITICAL_BETWEENNESS_QUANTILE * 100, threshold))
    print('articulation points       :', int(stops['is_articulation_point'].sum()))
    print('critical stops            : %d (%.1f%%)'
          % (int(stops['is_critical'].sum()), 100 * stops['is_critical'].mean()))
else:
    stops['is_critical'] = False
    print('WARNING: no betweenness / articulation-point columns in the metrics table;')
    print('         critical_stop_share will be 0 everywhere and should be ignored.')

         critical_stop_share will be 0 everywhere and should be ignored.


## External data dependency: the CBS 2021 socioeconomic layer

> **Data-dependency risk.** The socioeconomic index is not bundled with this repository. It's downloaded at run time from a public ArcGIS REST feature service hosted for the Central Bureau of Statistics (`services2.arcgis.com/.../SOEC_Stat11_2021/FeatureServer/27`, about 2,900 statistical-area polygons). If the service is renamed, retired, or blocked by a firewall, this notebook can't reproduce its inputs and there's no offline substitute in the repository. So the first successful run caches the raw download to `outputs/nb/09_socioeconomic_equity/data/cbs_socioeconomic_areas_2021.geojson`; every later run reads from the cache and needs no network at all. Keep that cached file - it's the only offline copy.
>
> The layer has already changed once: the fields `CLUSTER_2021` and `INDEX_VALUE_2021` were populated when the original analysis ran, but they're now served entirely as `NULL`. So the socioeconomic cluster is read from `eshkol_mad` (present for every polygon), and the continuous index value may come back empty - the notebook checks and reports this rather than assuming.

The service caps a response at 2,000 features, so the download pages through the layer using `resultOffset`. Only the few fields the analysis needs are requested, and every page is checked against an ArcGIS error envelope before decoding.

In [9]:
SOCIO_FIELDS = ['SEMEL_YISH', 'STAT11', 'YISHUV_STA', 'Shem_Yishuv', 'Shem_Yishuv_English',
                'Pop_Total', 'eshkol_mad', 'CLUSTER_2021', 'INDEX_VALUE_2021']

def cbs_query_url(offset=None, count_only=False):
    params = {'where': '1=1', 'f': 'json' if count_only else 'geojson'}
    if count_only:
        params['returnCountOnly'] = 'true'
    else:
        params.update({
            'outFields': ','.join(SOCIO_FIELDS),
            'returnGeometry': 'true',
            'outSR': '4326',
            'resultRecordCount': str(CBS_PAGE_SIZE),
            'resultOffset': str(offset or 0),
        })
    return CBS_LAYER_URL + '/query?' + urlencode(params, safe=',')

def _get_json(url):
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and 'error' in payload:
        raise RuntimeError('CBS service returned an error: ' + json.dumps(payload['error'], ensure_ascii=False))
    return payload

def download_cbs_layer():
    if CBS_CACHE.exists():
        print('Using the cached CBS download (no network needed):', CBS_CACHE)
        return gpd.read_file(CBS_CACHE)
    total = int(_get_json(cbs_query_url(count_only=True))['count'])
    print('CBS layer reports %d statistical areas; downloading in pages of %d...' % (total, CBS_PAGE_SIZE))
    pages = []
    for offset in range(0, total, CBS_PAGE_SIZE):
        print('  features %d-%d / %d' % (offset + 1, min(offset + CBS_PAGE_SIZE, total), total))
        payload = _get_json(cbs_query_url(offset=offset))
        pages.append(gpd.GeoDataFrame.from_features(payload['features'], crs='EPSG:4326'))
    raw = gpd.GeoDataFrame(pd.concat(pages, ignore_index=True), geometry='geometry', crs='EPSG:4326')
    try:
        raw.to_file(CBS_CACHE, driver='GeoJSON')
        print('Cached the raw layer to', CBS_CACHE)
    except Exception as exc:
        print('WARNING: could not cache the layer (' + str(exc) + '); the next run will download again.')
    return raw

raw_cbs = download_cbs_layer()
print('downloaded polygons:', len(raw_cbs))
print('fields             :', [c for c in raw_cbs.columns if c != 'geometry'])

CBS layer reports 2885 statistical areas; downloading in pages of 2000...
  features 1-2000 / 2885


  features 2001-2885 / 2885


Cached the raw layer to C:\Users\Sean-PC\Desktop\LAST SEMESTER\אלגוריתמים ברשתות\פרויקט גמר\outputs\nb\09_socioeconomic_equity\data\cbs_socioeconomic_areas_2021.geojson
downloaded polygons: 2885
fields             : ['SEMEL_YISH', 'STAT11', 'YISHUV_STA', 'Shem_Yishuv', 'Shem_Yishuv_English', 'Pop_Total', 'eshkol_mad', 'CLUSTER_2021', 'INDEX_VALUE_2021']


## Normalizing the socioeconomic layer

The service exposes the same attribute under several names (Hebrew and English aliases, 2019 and 2021 versions), so each analysis field is taken from the first source column that actually carries data. The rows are then restricted to polygons with a valid socioeconomic cluster in the range 1-10 and a real geometry; polygons without a cluster (non-residential areas, a few new towns) can't contribute to an equity comparison.

Every identifier is kept as a float or a plain string rather than pandas' nullable `Int64` type, because masked integer columns throw errors in `pd.cut`, in boolean masking, and inside `scipy` - a real source of failures in the original script.

In [10]:
def first_existing(frame, columns):
    """First column that exists, filled in from the later ones where it is null."""
    values = None
    for col in columns:
        if col in frame.columns:
            values = frame[col].copy() if values is None else values.where(values.notna(), frame[col])
    if values is None:
        return pd.Series(np.nan, index=frame.index, dtype='object')
    return values

socio = gpd.GeoDataFrame({
    'socio_locality_code': first_existing(raw_cbs, ['SEMEL_YISH', 'סמל_יישוב']),
    'socio_stat_area_code': first_existing(raw_cbs, ['STAT11', 'סמל_אזור_סטטיסטי']),
    'socio_unit_full_code': first_existing(raw_cbs, ['YISHUV_STA']),
    'socio_locality': first_existing(raw_cbs, ['Shem_Yishuv', 'שם_יישוב']),
    'socio_locality_en': first_existing(raw_cbs, ['Shem_Yishuv_English']),
    'socio_population': first_existing(raw_cbs, ['Pop_Total', 'אוכלוסיית_המדד']),
    'socio_cluster': first_existing(raw_cbs, ['eshkol_mad', 'CLUSTER_2021', 'אשכול']),
    'socio_index_value': first_existing(raw_cbs, ['INDEX_VALUE_2021', 'ערך_מדד']),
}, geometry=raw_cbs.geometry, crs=raw_cbs.crs)

for col in ['socio_locality_code', 'socio_stat_area_code', 'socio_unit_full_code',
            'socio_population', 'socio_cluster', 'socio_index_value']:
    socio[col] = pd.to_numeric(socio[col], errors='coerce')

def code_to_str(series):
    return series.map(lambda v: '' if pd.isna(v) else str(int(v)))

socio['socio_unit_id'] = code_to_str(socio['socio_unit_full_code'])
fallback_id = code_to_str(socio['socio_locality_code']) + '_' + code_to_str(socio['socio_stat_area_code'])
no_full_code = socio['socio_unit_full_code'].isna()
socio.loc[no_full_code, 'socio_unit_id'] = fallback_id[no_full_code]

keep = socio['socio_cluster'].between(1, 10).fillna(False) & socio.geometry.notna()
socio = socio.loc[keep].copy()

HAS_INDEX_VALUE = bool(socio['socio_index_value'].notna().any())
print('usable statistical areas   :', len(socio))
print('with a population figure   : %d (%.1f%%)'
      % (int(socio['socio_population'].notna().sum()), 100 * socio['socio_population'].notna().mean()))
print('continuous index available :', HAS_INDEX_VALUE)
if not HAS_INDEX_VALUE:
    print('  -> the service now serves INDEX_VALUE_2021 as NULL; only the 1-10 cluster is usable.')
print()
print(socio['socio_cluster'].value_counts().sort_index().to_string())

usable statistical areas   : 2721
with a population figure   : 2683 (98.6%)
continuous index available : False
  -> the service now serves INDEX_VALUE_2021 as NULL; only the 1-10 cluster is usable.

socio_cluster
1      79
2     225
3     232
4     166
5     363
6     301
7     631
8     366
9     347
10     11


## Matching stops to statistical areas

Two passes, in this order:

1. Point in polygon (`predicate='within'`) - the exact match; a stop inside a statistical area belongs to it.
2. Nearest polygon, capped at `NEAREST_JOIN_MAX_DISTANCE_M`, for stops that don't fall in any polygon. The CBS layer covers residential areas only, so stops on intercity roads, in industrial zones, and at interchanges legitimately fall outside every polygon; matching them to the nearest neighborhood is a documented approximation, and the distance is kept per stop (`socio_join_distance_m`) so the approximation can be filtered out later.

Distances are measured after projecting to EPSG:3857, which is roughly metric at Israel's latitude (a scale inflation of about 1.18x) - so the cap is conservative, not exact. The join-quality dictionary records the match rate and the distance distribution.

In [11]:
stops_gdf = gpd.GeoDataFrame(stops.copy(),
                             geometry=gpd.points_from_xy(stops['lon'], stops['lat']),
                             crs='EPSG:4326')

socio_cols = ['socio_unit_id', 'socio_locality_code', 'socio_stat_area_code', 'socio_locality',
              'socio_locality_en', 'socio_population', 'socio_cluster', 'socio_index_value', 'geometry']

joined = gpd.sjoin(stops_gdf, socio[socio_cols], how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')].drop(columns=['index_right'], errors='ignore')
joined['socio_join_method'] = np.where(joined['socio_cluster'].notna(), 'within', None)
joined['socio_join_distance_m'] = np.where(joined['socio_cluster'].notna(), 0.0, np.nan)

missing_index = joined.index[joined['socio_cluster'].isna()]
print('stops inside a polygon      :', int((joined['socio_join_method'] == 'within').sum()))
print('stops needing nearest join  :', len(missing_index))

if len(missing_index):
    nearest = gpd.sjoin_nearest(
        stops_gdf.loc[missing_index].to_crs(3857),
        socio[socio_cols].to_crs(3857),
        how='left',
        max_distance=NEAREST_JOIN_MAX_DISTANCE_M,
        distance_col='socio_join_distance_m',
    )
    nearest = nearest[~nearest.index.duplicated(keep='first')].drop(columns=['index_right'], errors='ignore')
    matched_nearest = nearest.index[nearest['socio_cluster'].notna()]
    for col in [c for c in socio_cols if c != 'geometry'] + ['socio_join_distance_m']:
        joined.loc[matched_nearest, col] = nearest.loc[matched_nearest, col]
    joined.loc[matched_nearest, 'socio_join_method'] = 'nearest'

joined['socio_cluster'] = pd.to_numeric(joined['socio_cluster'], errors='coerce')
joined['socio_cluster_group'] = pd.cut(joined['socio_cluster'], bins=[0, 4, 7, 10],
                                       labels=['low_1_4', 'middle_5_7', 'high_8_10'])

nearest_dist = joined.loc[joined['socio_join_method'] == 'nearest', 'socio_join_distance_m']
join_quality = {
    'stop_metrics_source': str(STOP_METRICS_PATH),
    'cbs_layer': CBS_LAYER_URL,
    'cbs_layer_is_external_runtime_dependency': True,
    'total_stops': int(len(joined)),
    'matched_within_polygon': int((joined['socio_join_method'] == 'within').sum()),
    'matched_by_nearest_polygon': int((joined['socio_join_method'] == 'nearest').sum()),
    'unmatched': int(joined['socio_join_method'].isna().sum()),
    'nearest_join_max_distance_m': NEAREST_JOIN_MAX_DISTANCE_M,
    'nearest_join_median_distance_m': None if nearest_dist.empty else round(float(nearest_dist.median()), 1),
    'nearest_join_max_observed_distance_m': None if nearest_dist.empty else round(float(nearest_dist.max()), 1),
}
join_quality['match_rate_pct'] = round(
    100 * (join_quality['matched_within_polygon'] + join_quality['matched_by_nearest_polygon'])
    / max(join_quality['total_stops'], 1), 2)

print()
print(json.dumps(join_quality, ensure_ascii=False, indent=2))

stops inside a polygon      : 23592
stops needing nearest join  : 6871



{
  "stop_metrics_source": "C:\\Users\\Sean-PC\\Desktop\\LAST SEMESTER\\אלגוריתמים ברשתות\\פרויקט גמר\\outputs\\nb\\04_centrality_analysis\\tables\\stop_metrics.csv",
  "cbs_layer": "https://services2.arcgis.com/xMRYm7cNgdR5RN6F/arcgis/rest/services/SOEC_Stat11_2021/FeatureServer/27",
  "cbs_layer_is_external_runtime_dependency": true,
  "total_stops": 30463,
  "matched_within_polygon": 23592,
  "matched_by_nearest_polygon": 6468,
  "unmatched": 403,
  "nearest_join_max_distance_m": 3000,
  "nearest_join_median_distance_m": 521.9,
  "nearest_join_max_observed_distance_m": 2999.7,
  "match_rate_pct": 98.68
}


## Aggregating to neighborhoods (CBS statistical areas)

The statistical area is the right unit of analysis, for two reasons:

1. The socioeconomic cluster is an attribute of the area, not of the stop. Correlating it at the stop level creates pseudo-replication: a dense city-center area with 90 stops would be counted 90 times against a suburb with 4 stops, purely because it has more stops.
2. Service per capita only exists at the area level, because the population figure is attached to the area.

For each area we count stops, sum scheduled trip stops, take the mean trip-graph degree and the share of critical stops, and record the dominant Louvain community / town / region / metro by mode. `stop_use_per_1000` - scheduled stops per 1,000 residents - is the service variable used from here on.

In [12]:
matched = joined[joined['socio_cluster'].notna()].copy()
matched['active_stop'] = matched['stop_use_count'] > 0
matched['low_socio_stop'] = matched['socio_cluster'] <= 4

nb = (matched.groupby('socio_unit_id')
      .agg(socio_cluster=('socio_cluster', 'first'),
           population=('socio_population', 'first'),
           stops=('stop_id', 'count'),
           active_stops=('active_stop', 'sum'),
           total_stop_use=('stop_use_count', 'sum'),
           avg_trip_graph_degree=('trip_graph_degree', 'mean'),
           critical_stops=('is_critical', 'sum'),
           community=('community', mode_or_na),
           city=('socio_locality', mode_or_na),
           region=('region', mode_or_na),
           metro=('metro', mode_or_na))
      .reset_index())

nb['socio_cluster'] = pd.to_numeric(nb['socio_cluster'], errors='coerce')
nb['community'] = pd.to_numeric(nb['community'], errors='coerce')
nb['stop_use_per_1000'] = per_1000(nb['total_stop_use'], nb['population'])
nb['stops_per_1000'] = per_1000(nb['stops'], nb['population'])
nb['stop_use_per_stop'] = np.where(nb['stops'] > 0, nb['total_stop_use'] / nb['stops'], np.nan)
nb['critical_stop_share'] = nb['critical_stops'] / nb['stops']
nb['active_stop_share'] = nb['active_stops'] / nb['stops']

print('neighbourhoods (statistical areas) reached by the network :', len(nb))
print('  with a usable per-capita figure                         :', int(nb['stop_use_per_1000'].notna().sum()))
print('  with a Louvain community                                :', int(nb['community'].notna().sum()))
nb.head()

neighbourhoods (statistical areas) reached by the network : 2681
  with a usable per-capita figure                         : 2643
  with a Louvain community                                : 2680


,socio_unit_id,socio_cluster,population,stops,active_stops,total_stop_use,avg_trip_graph_degree,critical_stops,community,city,region,metro,stop_use_per_1000,stops_per_1000,stop_use_per_stop,critical_stop_share,active_stop_share
0,100001,4.0,452.0,2,2,181,5.0000,0,6.0,תירוש,מרכז,פריפריה,400.442478,4.424779,90.5000,0.0,1.0
1,10150001,9.0,3155.0,10,10,6569,2.5000,0,63.0,מבשרת ציון,ירושלים,ירושלים,2082.091918,3.169572,656.9000,0.0,1.0
2,10150002,7.0,6081.0,32,32,11038,3.0625,0,63.0,מבשרת ציון,ירושלים,ירושלים,1815.161980,5.262292,344.9375,0.0,1.0
3,10150003,9.0,4305.0,16,16,6353,2.1250,0,63.0,מבשרת ציון,ירושלים,ירושלים,1475.725900,3.716609,397.0625,0.0,1.0
4,10150004,8.0,2272.0,16,16,11924,4.3750,0,63.0,מבשרת ציון,ירושלים,ירושלים,5248.239437,7.042254,745.2500,0.0,1.0


## The national picture

First the naive question: aggregated across the whole country, does service per capita depend on the socioeconomic cluster? Two views are produced.

- Summary by CBS cluster (1-10): population, stops, trip stops, and the service per capita of the median neighborhood. The medians are reported alongside the aggregate ratios because a handful of neighborhoods with central stops and tiny resident populations dominate any mean.
- Spearman correlation table at two scopes: neighborhood level (the defensible one) and stop level (kept for comparison with the earlier work, and flagged as pseudo-replicated).

Spearman is used rather than Pearson because the socioeconomic cluster is an ordinal scale in the range 1-10 and service per capita is extremely right-skewed.

In [13]:
cluster_summary = (nb.dropna(subset=['socio_cluster'])
                   .groupby('socio_cluster')
                   .agg(neighborhoods=('socio_unit_id', 'count'),
                        population=('population', 'sum'),
                        stops=('stops', 'sum'),
                        active_stops=('active_stops', 'sum'),
                        total_stop_use=('total_stop_use', 'sum'),
                        median_stops_per_1000=('stops_per_1000', 'median'),
                        median_stop_use_per_1000=('stop_use_per_1000', 'median'),
                        mean_trip_graph_degree=('avg_trip_graph_degree', 'mean'),
                        mean_critical_stop_share=('critical_stop_share', 'mean'))
                   .reset_index())
cluster_summary['stops_per_1000_residents'] = per_1000(cluster_summary['stops'], cluster_summary['population'])
cluster_summary['stop_use_per_1000_residents'] = per_1000(cluster_summary['total_stop_use'], cluster_summary['population'])
cluster_summary['stop_use_per_stop'] = np.where(cluster_summary['stops'] > 0,
                                                cluster_summary['total_stop_use'] / cluster_summary['stops'], np.nan)
cluster_summary['active_stop_share'] = cluster_summary['active_stops'] / cluster_summary['stops']

national_rows = []
for metric in ['stop_use_per_1000', 'stops_per_1000', 'stop_use_per_stop',
               'avg_trip_graph_degree', 'critical_stop_share']:
    rho, p_value, n = spearman(nb['socio_cluster'], nb[metric])
    national_rows.append({'scope': 'neighborhood (statistical area)', 'metric': metric,
                          'spearman_rho': rho, 'p_value': p_value, 'n': n,
                          'direction': describe_rho(rho)})
for metric in ['stop_use_count', 'trip_graph_degree', 'is_critical']:
    rho, p_value, n = spearman(matched['socio_cluster'], matched[metric].astype(float))
    national_rows.append({'scope': 'stop (pseudo-replicated)', 'metric': metric,
                          'spearman_rho': rho, 'p_value': p_value, 'n': n,
                          'direction': describe_rho(rho)})
national_correlations = pd.DataFrame(national_rows)

NATIONAL_RHO, NATIONAL_P, NATIONAL_N = spearman(nb['socio_cluster'], nb['stop_use_per_1000'])
print('NATIONAL pooled rho(socioeconomic cluster, stop calls per 1,000) = %.3f  (p = %.3g, n = %d)'
      % (NATIONAL_RHO, NATIONAL_P, NATIONAL_N))
print('direction:', describe_rho(NATIONAL_RHO), '- but note how small the effect is.')
print()
print(national_correlations.to_string(index=False))
print()
print(cluster_summary[['socio_cluster', 'neighborhoods', 'population', 'stops',
                       'median_stops_per_1000', 'median_stop_use_per_1000']].to_string(index=False))

NATIONAL pooled rho(socioeconomic cluster, stop calls per 1,000) = -0.151  (p = 6.33e-15, n = 2643)
direction: weaker neighbourhoods favoured - but note how small the effect is.

                          scope                metric  spearman_rho      p_value     n                        direction
neighborhood (statistical area)     stop_use_per_1000     -0.150869 6.330770e-15  2643   weaker neighbourhoods favoured
neighborhood (statistical area)        stops_per_1000      0.030380 1.184154e-01  2643 stronger neighbourhoods favoured
neighborhood (statistical area)     stop_use_per_stop     -0.174581 8.600503e-20  2681   weaker neighbourhoods favoured
neighborhood (statistical area) avg_trip_graph_degree     -0.040210 3.735227e-02  2681   weaker neighbourhoods favoured
neighborhood (statistical area)   critical_stop_share           NaN          NaN  2681                        undefined
       stop (pseudo-replicated)        stop_use_count     -0.110976 5.400965e-83 30060   weaker neigh

### Supporting figure: service by socioeconomic cluster

A plain descriptive chart of the table above. It shows neighborhood-median bars rather than the aggregate ratio, because the aggregate is dominated by a few neighborhoods that host central stops while almost no one lives in them. The bars are close to flat, and that's exactly the gist of the national story: at this resolution the network looks broadly equitable.

In [14]:
plot_data = cluster_summary.dropna(subset=['socio_cluster']).copy()
plot_data['socio_cluster'] = plot_data['socio_cluster'].astype(int)
palette = sns.color_palette('crest', n_colors=len(plot_data))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(plot_data['socio_cluster'], plot_data['median_stops_per_1000'], color=palette)
axes[0].set_title('Median neighbourhood: stops per 1,000 residents')
axes[0].set_xlabel('CBS socioeconomic cluster (1 = weakest, 10 = strongest)')
axes[0].set_ylabel('Stops / 1,000 residents')
axes[1].bar(plot_data['socio_cluster'], plot_data['median_stop_use_per_1000'], color=palette)
axes[1].set_title('Median neighbourhood: scheduled stop calls per 1,000 residents')
axes[1].set_xlabel('CBS socioeconomic cluster (1 = weakest, 10 = strongest)')
axes[1].set_ylabel('Stop calls / 1,000 residents')
for ax in axes:
    ax.set_xticks(plot_data['socio_cluster'])
fig.suptitle('National view: service level by socioeconomic cluster (medians over neighbourhoods)')
fig.tight_layout()
save_fig(fig, 'socioeconomic_access_by_cluster.png')

saved

 socioeconomic_access_by_cluster.png


WindowsPath('C:/Users/Sean-PC/Desktop/LAST SEMESTER/אלגוריתמים ברשתות/פרויקט גמר/outputs/nb/09_socioeconomic_equity/figures/socioeconomic_access_by_cluster.png')

## Louvain communities as a data-driven proxy for cities and metros

To test the paradox we need groups within which the "strong neighborhood vs. weak neighborhood" comparison is fair - that is, groups served by one operational subnetwork. Administrative city boundaries aren't in the GTFS file, but Louvain communities of the trip graph are, and they line up well with cities and metropolitan corridors because most trips stay inside one of them.

So each community is summarized and labeled from the data: the label is the most common CBS town among the community's stops. The previous script hard-coded community IDs (community 34 = Netanya, 23 = Beit She'an), which quietly mislabels every figure the moment community detection is re-run with a different seed or a different graph. Nothing here depends on a numeric community ID.

In [15]:
with_community = matched[matched['community'].notna()].copy()

community_summary = (with_community.groupby('community')
                     .agg(stops=('stop_id', 'count'),
                          active_stops=('active_stop', 'sum'),
                          total_stop_use_count=('stop_use_count', 'sum'),
                          avg_stop_use_count=('stop_use_count', 'mean'),
                          avg_trip_graph_degree=('trip_graph_degree', 'mean'),
                          avg_socio_cluster=('socio_cluster', 'mean'),
                          median_socio_cluster=('socio_cluster', 'median'),
                          dominant_socio_cluster=('socio_cluster', mode_or_na),
                          low_socio_stop_share=('low_socio_stop', 'mean'),
                          dominant_city=('socio_locality', mode_or_na),
                          dominant_region=('region', mode_or_na),
                          dominant_metro=('metro', mode_or_na),
                          socio_areas=('socio_unit_id', 'nunique'))
                     .reset_index())

covered_population = (nb.dropna(subset=['community'])
                      .groupby('community')['population'].sum()
                      .rename('covered_population_estimate').reset_index())
community_summary = community_summary.merge(covered_population, on='community', how='left')
community_summary['active_stop_share'] = community_summary['active_stops'] / community_summary['stops']
community_summary['use_per_1000'] = per_1000(community_summary['total_stop_use_count'],
                                             community_summary['covered_population_estimate'])
community_summary['community_label'] = [
    ('%s (#%d)' % (city, int(cid))) if isinstance(city, str) else ('community #%d' % int(cid))
    for city, cid in zip(community_summary['dominant_city'], community_summary['community'])]
community_summary = community_summary.sort_values('stops', ascending=False).reset_index(drop=True)

print('communities with socioeconomic coverage:', len(community_summary))
print()
print(community_summary[['community_label', 'stops', 'socio_areas', 'avg_socio_cluster',
                         'use_per_1000', 'dominant_region']].head(12).to_string(index=False))

communities with socioeconomic coverage: 73

 community_label  stops  socio_areas  avg_socio_cluster  use_per_1000 dominant_region
    באר שבע (#0)   1313          106           5.511805   1409.206204            דרום
     נהרייה (#2)    842           71           5.794537    805.486873            צפון
קריית שמונה (#1)    828           99           5.864734    938.007196            צפון
      נתניה (#3)    820          124           6.314634   1409.795475            מרכז
       אשבל (#5)    806           58           5.145161   1699.564831            צפון
      עפולה (#4)    796           97           5.131910   1670.320254            צפון
   קריית גת (#6)    788          127           5.579949   1017.567668            מרכז
       רמלה (#7)    775           90           5.838710   1160.822770            מרכז
  נוף הגליל (#8)    741           34           3.958165   2588.910806            צפון
    ירושלים (#9)    663           65           5.624434   2676.281792         ירושלים
   כפר סב

## Story step 1 - one point per community: no visible rule

The first presentation figure. Each point is a whole community, placed by its average socioeconomic level and its average service per capita. The cloud has no usable slope: averaging a community mixes its strong and weak neighborhoods together, and the between-community differences that remain are driven by density and geography, not by wealth.

The dashed line is an OLS fit on all the plotted points, that is, exactly the rows behind the quoted Spearman rho. The y-axis is clipped at the 95th percentile so the cloud is readable; the clipping affects the display only, never the fit or the statistic.

Output: `figures/socioeconomic_cluster_average_flat.png`.

In [16]:
flat = community_summary.dropna(subset=['avg_socio_cluster', 'use_per_1000']).copy()
rho_flat, p_flat, n_flat = spearman(flat['avg_socio_cluster'], flat['use_per_1000'])
y_view = float(np.nanpercentile(flat['use_per_1000'], 95))

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(flat['avg_socio_cluster'], flat['use_per_1000'],
           s=np.clip(flat['stops'], 30, 600), alpha=0.55,
           color='#34557a', edgecolor='white', linewidth=0.6)
coef_flat = ols_slope(flat['avg_socio_cluster'], flat['use_per_1000'])
if coef_flat is not None:
    xs = np.array([flat['avg_socio_cluster'].min(), flat['avg_socio_cluster'].max()])
    ax.plot(xs, np.polyval(coef_flat, xs), color='#c0392b', linewidth=3, linestyle='--',
            label='OLS fit on all %d communities (untrimmed)' % len(flat))
ax.set_ylim(0, 1.05 * y_view)
ax.set_xlim(0.5, 10.5)
ax.set_title('Each point is one Louvain community (%d communities)' % len(flat),
             fontsize=16, fontweight='bold', pad=12)
ax.set_xlabel('Mean socioeconomic cluster of the community  (1 = weakest, 10 = strongest)')
ax.set_ylabel('Service per capita (scheduled stop calls / 1,000 residents)')
ax.legend(loc='upper left')
ax.annotate('Spearman rho = %.2f,  p = %.2f  (n = %d communities)' % (rho_flat, p_flat, n_flat)
            + '\nBubble size = stops in the community'
            + '\nY axis clipped at the 95th percentile for readability;'
            + '\nthe fit and the rho use every point.',
            xy=(0.98, 0.95), xycoords='axes fraction', ha='right', va='top', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.5', fc='#f4f4f4', ec='#bbbbbb', lw=1.0))
save_fig(fig, 'socioeconomic_cluster_average_flat.png')
print('community-level rho = %.3f (p = %.3f) -> %s'
      % (rho_flat, p_flat, 'significant' if p_flat < ALPHA else 'NOT significant'))

saved socioeconomic_cluster_average_flat.png
community-level rho = -0.182 (p = 0.124) -> NOT significant


## The same point, told with cities named by name

The scatter above convinces a statistician and is invisible to everyone else, so the same fact is restated using familiar places: communities ordered from poorest to richest, with their service per capita on the bars. The bar heights jump around with no relation to the order.

**A choice.** The original script hard-coded six city names. Hard-coded choices are cherry-picking unless they're justified, so here the cities are taken from the data: the `N_CITIES_NO_RULE` communities with the largest number of stops, after removing duplicates by dominant city, and sorted by socioeconomic level. Set `CITIES_IN_NO_RULE_FIGURE` to a list of town names to reproduce a particular curated slide instead.

Output: `figures/socioeconomic_cities_no_rule.png`.

In [17]:
available = community_summary.dropna(subset=['avg_socio_cluster', 'use_per_1000', 'dominant_city'])
if CITIES_IN_NO_RULE_FIGURE:
    picked = (available[available['dominant_city'].isin(CITIES_IN_NO_RULE_FIGURE)]
              .drop_duplicates('dominant_city'))
    selection_note = 'Cities chosen by hand (CITIES_IN_NO_RULE_FIGURE)'
else:
    picked = (available.sort_values('stops', ascending=False)
              .drop_duplicates('dominant_city')
              .head(N_CITIES_NO_RULE))
    selection_note = 'The %d largest communities by stop count (no hand-picking)' % N_CITIES_NO_RULE
picked = picked.sort_values('avg_socio_cluster')

fig, ax = plt.subplots(figsize=(12, 7))
norm = plt.Normalize(1, 10)
colors = plt.cm.RdYlGn(norm(picked['avg_socio_cluster']))
bars = ax.bar(range(len(picked)), picked['use_per_1000'], color=colors,
              edgecolor='#333333', linewidth=0.8)
ax.set_xticks(range(len(picked)))
ax.set_xticklabels(['%s\n(cluster %.1f)' % (c, s)
                    for c, s in zip(picked['dominant_city'], picked['avg_socio_cluster'])],
                   fontsize=12)
for bar, val in zip(bars, picked['use_per_1000']):
    ax.text(bar.get_x() + bar.get_width() / 2, val, format(int(round(val)), ','),
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Service per capita (scheduled stop calls / 1,000 residents)')
ax.set_title('Ordered from the poorest community to the richest - service jumps with no rule',
             fontsize=15, fontweight='bold', pad=14)
ax.annotate('Socioeconomic level does not predict service\nSpearman rho = %.2f,  p = %.2f  (%s, all %d communities)'
            % (rho_flat, p_flat, 'significant' if p_flat < ALPHA else 'not significant', n_flat)
            + '\n' + selection_note,
            xy=(0.98, 0.95), xycoords='axes fraction', ha='right', va='top',
            fontsize=13, fontweight='bold', color='#c0392b',
            bbox=dict(boxstyle='round,pad=0.5', fc='#fdecea', ec='#c0392b', lw=1.5))
ax.margins(y=0.20)
save_fig(fig, 'socioeconomic_cities_no_rule.png')
print(picked[['community_label', 'stops', 'avg_socio_cluster', 'use_per_1000']].to_string(index=False))

saved socioeconomic_cities_no_rule.png
 community_label  stops  avg_socio_cluster  use_per_1000
      עפולה (#4)    796           5.131910   1670.320254
       אשבל (#5)    806           5.145161   1699.564831
    באר שבע (#0)   1313           5.511805   1409.206204
     נהרייה (#2)    842           5.794537    805.486873
קריית שמונה (#1)    828           5.864734    938.007196
      נתניה (#3)    820           6.314634   1409.795475


## Step 2 - the within-community test, with a correction for multiple comparisons

Now the real test. For each community with at least `MIN_NEIGHBORHOODS` neighborhoods spread across at least `MIN_DISTINCT_SOCIO_LEVELS` socioeconomic levels, we correlate the neighborhood's socioeconomic cluster with its service per capita. This is a within-group correlation: it compares a strong neighborhood to a weak neighborhood served by the same subnetwork, and that's the comparison the aggregate national number can't make.

**A bug fixed here: multiple comparisons.** About 50-60 communities are tested at once. At a significance level of alpha = 0.05 that's enough to produce, in expectation, about three "significant" results from noise alone, and the previous version reported the raw p-values with no correction at all. This notebook reports, side by side:

- `p_use_per_capita` - the raw p-value;
- `p_fdr_bh` - corrected by Benjamini-Hochberg (controls the false discovery rate, the right correction for screening of this kind);
- `p_bonferroni` - corrected by Bonferroni (controls the family-wise error rate; conservative).

The figures mark significance using the FDR-corrected value, and the raw p-value stays in the table so nothing is hidden. Benjamini-Hochberg is implemented in the code itself (a step-up over the sorted p-values with a monotonicity pass) to avoid an extra dependency.

In [18]:
def benjamini_hochberg(p_values):
    """Benjamini-Hochberg FDR-adjusted p-values; NaNs pass through as NaN."""
    p = np.asarray(p_values, dtype=float)
    out = np.full(p.shape, np.nan)
    ok = ~np.isnan(p)
    pv = p[ok]
    m = pv.size
    if m == 0:
        return out
    order = np.argsort(pv)
    ranked = pv[order]
    adjusted = ranked * m / np.arange(1, m + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]   # enforce monotonicity
    adjusted = np.clip(adjusted, 0, 1)
    restored = np.empty(m)
    restored[order] = adjusted
    out[ok] = restored
    return out

rows = []
for community, group in nb.dropna(subset=['community']).groupby('community'):
    group = group.dropna(subset=['socio_cluster'])
    if len(group) < MIN_NEIGHBORHOODS or group['socio_cluster'].nunique() < MIN_DISTINCT_SOCIO_LEVELS:
        continue
    rho_use, p_use, n_use = spearman(group['socio_cluster'], group['stop_use_per_1000'])
    rho_stops, p_stops, _ = spearman(group['socio_cluster'], group['stops_per_1000'])
    stop_group = with_community[with_community['community'] == community]
    rho_stop_level, p_stop_level, _ = spearman(stop_group['socio_cluster'], stop_group['stop_use_count'])
    city = mode_or_na(group['city'])
    rows.append({
        'community': int(community),
        'dominant_city': city,
        'community_label': ('%s (#%d)' % (city, int(community))) if isinstance(city, str) else ('community #%d' % int(community)),
        'dominant_region': mode_or_na(group['region']),
        'dominant_metro': mode_or_na(group['metro']),
        'n_neighborhoods': int(len(group)),
        'n_neighborhoods_with_population': int(n_use),
        'n_stops': int(len(stop_group)),
        'population': float(group['population'].sum()),
        'mean_socio_cluster': float(group['socio_cluster'].mean()),
        'socio_cluster_spread': int(group['socio_cluster'].nunique()),
        'rho_use_per_capita': rho_use,
        'p_use_per_capita': p_use,
        'rho_stops_per_capita': rho_stops,
        'p_stops_per_capita': p_stops,
        'rho_stop_level_use': rho_stop_level,
        'p_stop_level_use': p_stop_level,
    })

within = pd.DataFrame(rows)
if within.empty:
    raise RuntimeError('No community passed the within-community thresholds - check the community coverage above.')

N_TESTS = int(within['p_use_per_capita'].notna().sum())
within['abs_rho'] = within['rho_use_per_capita'].abs()
within['p_fdr_bh'] = benjamini_hochberg(within['p_use_per_capita'])
within['p_bonferroni'] = np.clip(within['p_use_per_capita'] * N_TESTS, 0, 1)
within['significant_raw_05'] = within['p_use_per_capita'] < ALPHA
within['significant_fdr_05'] = within['p_fdr_bh'] < ALPHA
within['significant_bonferroni_05'] = within['p_bonferroni'] < ALPHA
within['direction'] = [describe_rho(r) for r in within['rho_use_per_capita']]
within = within.sort_values('rho_use_per_capita').reset_index(drop=True)

valid = within.dropna(subset=['rho_use_per_capita'])
print('communities tested                       : %d' % N_TESTS)
print('rho range                                : %.2f .. %.2f'
      % (valid['rho_use_per_capita'].min(), valid['rho_use_per_capita'].max()))
print('significant at raw p < %.2f               : %d' % (ALPHA, int(within['significant_raw_05'].sum())))
print('significant after Benjamini-Hochberg FDR : %d' % int(within['significant_fdr_05'].sum()))
print('significant after Bonferroni             : %d' % int(within['significant_bonferroni_05'].sum()))
print('expected false positives at raw alpha    : %.1f' % (ALPHA * N_TESTS))
print()
print('Strongest negative rho (service tilted to the weaker neighbourhoods):')
print(valid.head(5)[['community_label', 'dominant_region', 'n_neighborhoods',
                     'rho_use_per_capita', 'p_use_per_capita', 'p_fdr_bh']].to_string(index=False))
print()
print('Strongest positive rho (service tilted to the stronger neighbourhoods):')
print(valid.tail(5)[['community_label', 'dominant_region', 'n_neighborhoods',
                     'rho_use_per_capita', 'p_use_per_capita', 'p_fdr_bh']].to_string(index=False))

communities tested                       : 57
rho range                                : -0.69 .. 0.43
significant at raw p < 0.05               : 16
significant after Benjamini-Hochberg FDR : 8
significant after Bonferroni             : 3
expected false positives at raw alpha    : 2.9

Strongest negative rho (service tilted to the weaker neighbourhoods):
    community_label dominant_region  n_neighborhoods  rho_use_per_capita  p_use_per_capita  p_fdr_bh
   ביתר עילית (#44)            מרכז               17           -0.691333          0.003015  0.025950
מודיעין עילית (#45)            מרכז               21           -0.651886          0.002492  0.025950
         יבנה (#52)            מרכז               23           -0.627212          0.001359  0.019363
         נתניה (#3)            מרכז              102           -0.426773          0.000010  0.000543
       נתיבות (#13)            דרום               73           -0.415246          0.000259  0.007386

Strongest positive rho (service til

## Figure: within-community correlations against the national line

One bar per community, showing the raw Spearman rho (see the sign-convention section: negative = service tilts toward weaker neighborhoods, green; positive = tilts toward stronger ones, red). The dashed blue line is the aggregate national rho, which sits near zero while individual communities reach far in both directions - that gap is the paradox.

Two integrity notes are drawn on the figure itself:
- The bars are faded when the community isn't significant after FDR correction, so a reader won't mistake a small, noisy community for a finding.
- The panel shows the `TOP_COMMUNITIES_FOR_FIGURE` communities with the largest absolute rho among those with at least `FIG_MIN_NEIGHBORHOODS` neighborhoods. That's a selection on effect size, so it exaggerates the typical spread; the full, unselected list is in `tables/socioeconomic_within_cluster_correlation.csv`.

Output: `figures/socioeconomic_within_cluster_correlation.png`.

In [19]:
bar_data = valid[valid['n_neighborhoods'] >= FIG_MIN_NEIGHBORHOODS].copy()
bar_data = bar_data.reindex(bar_data['abs_rho'].sort_values(ascending=False).index)
bar_data = bar_data.head(TOP_COMMUNITIES_FOR_FIGURE).sort_values('rho_use_per_capita')

labels = ['%s  (%d neighbourhoods)' % (c, n)
          for c, n in zip(bar_data['dominant_city'], bar_data['n_neighborhoods'])]
colors = [rho_color(v) for v in bar_data['rho_use_per_capita']]

fig, ax = plt.subplots(figsize=(13, 8))
y = np.arange(len(bar_data))
bars = ax.barh(y, bar_data['rho_use_per_capita'], color=colors, edgecolor='white')
for bar, sig in zip(bars, bar_data['significant_fdr_05']):
    if not sig:
        bar.set_alpha(0.35)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=11)
ax.axvline(0, color='#555555', linewidth=1)
ax.axvline(NATIONAL_RHO, color='#2471a3', linestyle='--', linewidth=2.5,
           label='national pooled rho = %.2f' % NATIONAL_RHO)
for yi, v in enumerate(bar_data['rho_use_per_capita']):
    ax.text(v + (0.015 if v >= 0 else -0.015), yi, '%+.2f' % v, va='center',
            ha='left' if v >= 0 else 'right', fontsize=11, fontweight='bold')
lo = float(bar_data['rho_use_per_capita'].min())
hi = float(bar_data['rho_use_per_capita'].max())
ax.set_xlim(lo - 0.20, hi + 0.20)
ax.set_xlabel(RHO_AXIS_LABEL + '\nraw rho, no sign flipping   |   ' + RHO_NEG_MEANING
              + '   |   ' + RHO_POS_MEANING, fontsize=11)
ax.set_title('The socioeconomic service gap changes sign from community to community\n'
             'green = weaker neighbourhoods get more service per capita, red = stronger ones do',
             fontsize=14, pad=12)
ax.legend(loc='lower right', fontsize=11)
ax.text(0.02, 0.97,
        'Faded bar = not significant after Benjamini-Hochberg FDR correction\n'
        '(%d of %d tested communities survive FDR; raw p < %.2f would pass %d)\n'
        'Panel shows the %d largest |rho| with >= %d neighbourhoods - a selection on effect size'
        % (int(within['significant_fdr_05'].sum()), N_TESTS, ALPHA,
           int(within['significant_raw_05'].sum()), len(bar_data), FIG_MIN_NEIGHBORHOODS),
        transform=ax.transAxes, ha='left', va='top', fontsize=11, color='#333333',
        bbox=dict(boxstyle='round,pad=0.4', fc='#f4f4f4', ec='#cccccc', lw=1.0))
save_fig(fig, 'socioeconomic_within_cluster_correlation.png')

saved

 socioeconomic_within_cluster_correlation.png


WindowsPath('C:/Users/Sean-PC/Desktop/LAST SEMESTER/אלגוריתמים ברשתות/פרויקט גמר/outputs/nb/09_socioeconomic_equity/figures/socioeconomic_within_cluster_correlation.png')

## Figure: three example communities (selection extremes, not typical cases)

Three scatter panels, one neighborhood per point. The panels are chosen as the most negative rho, the rho closest to zero, and the most positive rho among communities with at least `EXAMPLE_MIN_NEIGHBORHOODS` neighborhoods.

**Read this as a selection extreme, not a representative sample.** Picking the argmin and argmax of a statistic across about 50 tests guarantees strong-looking panels even under pure noise; that's exactly why the FDR-corrected p-value is printed in each panel's title. The figure's job is to show what a strong within-community relationship looks like, not to estimate how common one is.

Each red line is an OLS fit on all of the community's neighborhoods - the exact same rows as the rho in the title. The y-axis is clipped for readability only.

Output: `figures/socioeconomic_within_cluster_examples.png`.

In [20]:
pool = valid[valid['n_neighborhoods'] >= EXAMPLE_MIN_NEIGHBORHOODS]
if len(pool) < 3:
    print('Only %d communities have >= %d neighbourhoods; falling back to >= %d.'
          % (len(pool), EXAMPLE_MIN_NEIGHBORHOODS, FIG_MIN_NEIGHBORHOODS))
    pool = valid[valid['n_neighborhoods'] >= FIG_MIN_NEIGHBORHOODS]
pool = pool.sort_values('rho_use_per_capita')

most_negative = pool.iloc[0]
most_positive = pool.iloc[-1]
near_zero = pool.reindex(pool['rho_use_per_capita'].abs().sort_values().index).iloc[0]
picks = [('most negative rho', most_negative), ('rho closest to zero', near_zero),
         ('most positive rho', most_positive)]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.4))
for ax, (why, row) in zip(axes, picks):
    group = nb[nb['community'] == row['community']].dropna(subset=['socio_cluster', 'stop_use_per_1000'])
    ax.scatter(group['socio_cluster'], group['stop_use_per_1000'],
               s=np.clip(group['population'] / 200, 12, 240), alpha=0.65,
               color='#2c3e50', edgecolor='white', linewidth=0.4)
    coef = ols_slope(group['socio_cluster'], group['stop_use_per_1000'])
    if coef is not None:
        xs = np.array([group['socio_cluster'].min(), group['socio_cluster'].max()])
        ax.plot(xs, np.polyval(coef, xs), color='#c0392b', linewidth=2)
    y_cap = float(np.nanpercentile(group['stop_use_per_1000'], 97))
    ax.set_ylim(-0.05 * y_cap, 1.10 * y_cap)
    ax.set_title('%s\nrho = %.2f   (p_raw = %.3g, p_FDR = %.3g, n = %d)\n[selected as: %s]'
                 % (row['community_label'], row['rho_use_per_capita'], row['p_use_per_capita'],
                    row['p_fdr_bh'], int(row['n_neighborhoods']), why), fontsize=11)
    ax.set_xlabel('Socioeconomic cluster (1 = weakest, 10 = strongest)')
    ax.set_ylabel('Stop calls per 1,000 residents')
fig.suptitle('Inside a single community: strong vs weak neighbourhoods and their service level\n'
             'these three panels are SELECTION EXTREMA (argmin / nearest-zero / argmax of rho), not typical communities',
             fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.92))
save_fig(fig, 'socioeconomic_within_cluster_examples.png')

saved socioeconomic_within_cluster_examples.png


WindowsPath('C:/Users/Sean-PC/Desktop/LAST SEMESTER/אלגוריתמים ברשתות/פרויקט גמר/outputs/nb/09_socioeconomic_equity/figures/socioeconomic_within_cluster_examples.png')

## Figure: Simpson's paradox, in one chart

The bottom line. Two communities whose internal trends point in opposite directions are plotted over the aggregate national trend. The national line is nearly flat not because service is evenly distributed, but because opposite local gradients cancel each other out when the neighborhoods are averaged together. This is the classic definition of Simpson's paradox, and it's why a single national correlation is the wrong answer to the equity question.

The two communities are chosen from the data - the most negative rho and the most positive rho within a community among those with at least `FIG_MIN_NEIGHBORHOODS` neighborhoods - and labeled by their most common town. No community ID is hard-coded anywhere (the source hard-coded community 34 = Netanya and 23 = Beit She'an, which broke the moment community detection was re-run).

All three lines are OLS fits on untrimmed data, consistent with the quoted rho values. The y-axis display is clipped at the `DISPLAY_TRIM_PCT` percentile so the lines stay on screen, and `tables/trend_fit_sensitivity.csv` records how much each slope would have changed if that trimming had also been applied to the fit - the mismatch that the previous version had.

Output: `figures/socioeconomic_simpson_paradox.png`.

In [21]:
example_pool = valid[valid['n_neighborhoods'] >= FIG_MIN_NEIGHBORHOODS].sort_values('rho_use_per_capita')
simpson_picks = [(example_pool.iloc[0], '#2980b9'), (example_pool.iloc[-1], '#e67e22')]

all_nb = nb.dropna(subset=['socio_cluster', 'stop_use_per_1000'])
y_cap = float(np.nanpercentile(all_nb['stop_use_per_1000'], DISPLAY_TRIM_PCT))

fig, ax = plt.subplots(figsize=(12, 7))
xs = np.array([1.0, 10.0])
coef_nat = ols_slope(all_nb['socio_cluster'], all_nb['stop_use_per_1000'])
ax.plot(xs, np.polyval(coef_nat, xs), color='#222222', linewidth=3.5, linestyle='--',
        label='National, all %d neighbourhoods (rho = %+.2f) - the trends cancel out'
              % (NATIONAL_N, NATIONAL_RHO), zorder=5)

sensitivity_rows = []
def record_sensitivity(name, frame):
    full = ols_slope(frame['socio_cluster'], frame['stop_use_per_1000'])
    cap = float(np.nanpercentile(frame['stop_use_per_1000'], DISPLAY_TRIM_PCT))
    trimmed_frame = frame[frame['stop_use_per_1000'] <= cap]
    trimmed = ols_slope(trimmed_frame['socio_cluster'], trimmed_frame['stop_use_per_1000'])
    rho, p_value, n = spearman(frame['socio_cluster'], frame['stop_use_per_1000'])
    sensitivity_rows.append({
        'series': name,
        'n_untrimmed': int(len(frame)),
        'n_after_p%d_trim' % DISPLAY_TRIM_PCT: int(len(trimmed_frame)),
        'ols_slope_untrimmed_used_in_figures': None if full is None else float(full[0]),
        'ols_slope_trimmed_old_behaviour': None if trimmed is None else float(trimmed[0]),
        'spearman_rho_untrimmed': rho,
        'p_value_untrimmed': p_value,
    })

record_sensitivity('NATIONAL (all neighbourhoods)', all_nb)
for row, color in simpson_picks:
    group = nb[nb['community'] == row['community']].dropna(subset=['socio_cluster', 'stop_use_per_1000'])
    shown = group[group['stop_use_per_1000'] <= y_cap]
    ax.scatter(shown['socio_cluster'], shown['stop_use_per_1000'], s=70, alpha=0.55,
               color=color, edgecolor='white', linewidth=0.5, zorder=3)
    coef = ols_slope(group['socio_cluster'], group['stop_use_per_1000'])
    gx = np.array([group['socio_cluster'].min(), group['socio_cluster'].max()])
    ax.plot(gx, np.polyval(coef, gx), color=color, linewidth=4.5, zorder=4,
            label='%s  (rho = %+.2f, p_FDR = %.3g, n = %d)'
                  % (row['community_label'], row['rho_use_per_capita'], row['p_fdr_bh'],
                     int(row['n_neighborhoods'])))
    record_sensitivity(row['community_label'], group)

ax.set_ylim(0, 1.05 * y_cap)
ax.set_xlim(0.5, 10.5)
ax.set_xlabel('Socioeconomic cluster of the neighbourhood  (1 = weakest, 10 = strongest)')
ax.set_ylabel('Service per capita (scheduled stop calls / 1,000 residents)')
ax.set_title("Same country, opposite conclusion in each city - Simpson's paradox",
             fontsize=15, fontweight='bold', pad=12)
ax.legend(loc='upper right', fontsize=11, frameon=True, title='Trend fitted within:')
ax.text(0.02, 0.97,
        'Lines are OLS fits on untrimmed data, matching the quoted rho.\n'
        'Only the y-axis view is clipped (p%d) so every line stays on screen.' % DISPLAY_TRIM_PCT,
        transform=ax.transAxes, ha='left', va='top', fontsize=11, color='#333333',
        bbox=dict(boxstyle='round,pad=0.4', fc='#f4f4f4', ec='#cccccc', lw=1.0))
save_fig(fig, 'socioeconomic_simpson_paradox.png')

trend_sensitivity = pd.DataFrame(sensitivity_rows)
print(trend_sensitivity.to_string(index=False))

saved socioeconomic_simpson_paradox.png
                       series  n_untrimmed  n_after_p90_trim  ols_slope_untrimmed_used_in_figures  ols_slope_trimmed_old_behaviour  spearman_rho_untrimmed  p_value_untrimmed
NATIONAL (all neighbourhoods)         2643              2378                          -289.848497                       -60.725951               -0.150869       6.330770e-15
          מודיעין עילית (#45)           19                17                          -597.414959                      -292.407012               -0.651886       2.492017e-03
                ירושלים (#11)           57                51                           317.410097                       286.960888                0.431329       8.087372e-04


## Saving the stage outputs

Everything is written under `outputs/nb/09_socioeconomic_equity/`. The stop-level export deliberately carries `trip_graph_degree` and no column named `degree`, so a downstream stage can't quietly pick up the retired proximity-graph degree that appeared in the published `stops_with_socioeconomic.csv`. The `socioeconomic_summary.json` file records the national baseline together with the multiple-comparison counts, so the headline numbers can be quoted without re-running anything.

In [22]:
stop_export_cols = ['stop_id', 'stop_name', 'region', 'metro', 'community', 'lat', 'lon',
                    'trip_graph_degree', 'trip_graph_weighted_degree', 'stop_use_count',
                    'is_critical', 'socio_join_method', 'socio_join_distance_m', 'socio_unit_id',
                    'socio_locality', 'socio_locality_en', 'socio_cluster', 'socio_cluster_group',
                    'socio_index_value', 'socio_population']
stop_export_cols = [c for c in stop_export_cols if c in joined.columns]
stop_export = pd.DataFrame(joined[stop_export_cols])

tables = STAGE / 'tables'
stop_export.to_csv(tables / 'stops_with_socioeconomic.csv', index=False, encoding='utf-8-sig')
nb.to_csv(tables / 'socioeconomic_neighborhood_access.csv', index=False, encoding='utf-8-sig')
cluster_summary.to_csv(tables / 'socioeconomic_cluster_summary.csv', index=False, encoding='utf-8-sig')
community_summary.to_csv(tables / 'community_socioeconomic_summary.csv', index=False, encoding='utf-8-sig')
national_correlations.to_csv(tables / 'socioeconomic_national_correlations.csv', index=False, encoding='utf-8-sig')
within.to_csv(tables / 'socioeconomic_within_cluster_correlation.csv', index=False, encoding='utf-8-sig')
trend_sensitivity.to_csv(tables / 'trend_fit_sensitivity.csv', index=False, encoding='utf-8-sig')
(tables / 'socioeconomic_join_quality.json').write_text(
    json.dumps(join_quality, ensure_ascii=False, indent=2), encoding='utf-8')

summary = {
    'sign_convention': 'raw Spearman rho; negative = more service per capita in weaker neighbourhoods',
    'service_variable': 'scheduled stop calls per 1,000 residents, at CBS statistical-area level',
    'degree_source': 'trip-adjacency graph (the retired 500 m proximity degree is NOT used)',
    'national_neighborhood_rho': NATIONAL_RHO,
    'national_neighborhood_p': NATIONAL_P,
    'national_neighborhood_n': NATIONAL_N,
    'community_level_rho': rho_flat,
    'community_level_p': p_flat,
    'community_level_n': n_flat,
    'communities_tested': N_TESTS,
    'within_rho_min': float(valid['rho_use_per_capita'].min()),
    'within_rho_max': float(valid['rho_use_per_capita'].max()),
    'significant_raw': int(within['significant_raw_05'].sum()),
    'significant_fdr_bh': int(within['significant_fdr_05'].sum()),
    'significant_bonferroni': int(within['significant_bonferroni_05'].sum()),
    'expected_false_positives_at_raw_alpha': ALPHA * N_TESTS,
    'cbs_layer': CBS_LAYER_URL,
    'cbs_index_value_available': HAS_INDEX_VALUE,
    'stop_metrics_source': str(STOP_METRICS_PATH),
}
(tables / 'socioeconomic_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, default=float), encoding='utf-8')

print('tables written to', tables)
for path in sorted(tables.iterdir()):
    print('  ', path.name)
print()
print('figures written to', STAGE / 'figures')
for path in sorted((STAGE / 'figures').iterdir()):
    print('  ', path.name)

tables written to C:\Users\Sean-PC\Desktop\LAST SEMESTER\אלגוריתמים ברשתות\פרויקט גמר\outputs\nb\09_socioeconomic_equity\tables
   community_socioeconomic_summary.csv
   socioeconomic_cluster_summary.csv
   socioeconomic_join_quality.json
   socioeconomic_national_correlations.csv
   socioeconomic_neighborhood_access.csv
   socioeconomic_summary.json
   socioeconomic_within_cluster_correlation.csv
   stops_with_socioeconomic.csv
   trend_fit_sensitivity.csv

figures written to C:\Users\Sean-PC\Desktop\LAST SEMESTER\אלגוריתמים ברשתות\פרויקט גמר\outputs\nb\09_socioeconomic_equity\figures
   socioeconomic_access_by_cluster.png
   socioeconomic_cities_no_rule.png
   socioeconomic_cluster_average_flat.png
   socioeconomic_simpson_paradox.png
   socioeconomic_within_cluster_correlation.png
   socioeconomic_within_cluster_examples.png


## Conclusions

1. Nationally, socioeconomic status barely predicts transit service per capita. Aggregating every statistical area in the country gives a Spearman rho of about -0.15 - statistically significant only because the sample runs to thousands of neighborhoods, and far too weak to say anything about a specific place. Aggregating higher still, to whole Louvain communities, even that vanishes: the community-level correlation isn't significant.

2. Within a single community the relationship is usually strong, and its sign isn't stable. Across the roughly 50-60 communities large enough to test, the within-community rho ranges from about -0.7 to +0.5. Some metros clearly concentrate service per capita in their weaker neighborhoods; others do the opposite. Averaging them nationally cancels both effects - Simpson's paradox, and the headline result of this notebook.

3. The honest version of the finding is smaller than the raw version. Of the communities tested, noticeably fewer survive the Benjamini-Hochberg FDR correction than pass a raw p threshold of 0.05, and fewer still survive Bonferroni. With about 50 simultaneous tests, about three raw "significant" results are expected from noise alone. The paradox itself is stable - the spread of signs is much larger than sampling noise - but single-city claims should be quoted from the FDR column, not the raw column.

4. What this doesn't show. Scheduled stops per resident measure supply, not accessibility, not travel demand, and not travel time; a neighborhood on a busy arterial corridor scores high even if no line goes where its residents need to. Louvain communities are a proxy for cities and metros, not municipal boundaries, so a "city" label is the community's most common town, not an administrative fact. About a fifth of stops are matched to their statistical area through the nearest-polygon fallback rather than exact containment, and about 6% of CBS areas carry no population figure and so drop out of any per-capita statistic.

5. A reproducibility risk worth stating on submission. The socioeconomic data is pulled at run time from an external ArcGIS service and isn't bundled with this repository. The layer has already changed once - `INDEX_VALUE_2021` and `CLUSTER_2021` are now served as `NULL`, so only the `eshkol_mad` cluster in the range 1-10 is usable - which means an exact numeric reproduction of the earlier report isn't guaranteed. Keep the cached GeoJSON file in `outputs/nb/09_socioeconomic_equity/data/`.